Task-1: Predictive Modeling (Classification)

1. Importing Librabries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

2. Load the dataset

In [2]:
df = pd.read_csv('Stock Prices Data Set.csv')
df.dropna(inplace=True) # Handle missing values

# Inspect the target distribution
df['Target'] = (df['close'] > df['open']).astype(int)
df['Target'].value_counts()

1    255779
0    241682
Name: Target, dtype: int64

3. Feature Selection and Scaling

In [3]:
# Select features and target
features = ['open', 'high', 'low', 'volume']
X = df[features]
y = df['Target']

# Take a sample of 10,000 for faster training in this tutorial
df_sample = df.sample(10000, random_state=42)
X_sample = df_sample[features]
y_sample = df_sample['Target']

# Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.2, random_state=42)

# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

4. Training and Initial Model Evaluation

In [4]:
# Initialize multiple classification models
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

# Train and evaluate models
results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    }

pd.DataFrame(results).T

,Accuracy,Precision,Recall,F1-Score
Logistic Regression,0.5770,0.557541,0.996244,0.714960
Decision Tree,0.6045,0.633268,0.611268,0.622074
Random Forest,0.6760,0.697256,0.692019,0.694628


5. Hyperparameter Tuning using Grid Search

In [5]:
# Define parameters to search
param_grid = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

# Perform Grid Search
grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=3, scoring='f1')
grid_search.fit(X_train_scaled, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
best_dt = grid_search.best_estimator_
y_pred_tuned = best_dt.predict(X_test_scaled)

# Evaluate the tuned model
print(f"Tuned Decision Tree Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")

Best Parameters: {'max_depth': 20, 'min_samples_split': 2}
Tuned Decision Tree Accuracy: 0.5460
